# Day 30: Project 03, Expense Tracker

**Today's question:** how do we organise a program that is too big for one file, so that each part can be
tested on its own?

The brief is in `projects/03-expense-tracker/README.md`. This notebook shows the **design** and a worked
example on a *different* kind of data: a reading log. You will apply the same ideas to expenses.
Before running a cell marked 🤔, predict what it will show.

## 1. The plan: three modules, three jobs

```
expenses.txt  <-- storage.py -->  list of expense dicts  --> reports.py --> totals, largest, month
                     ^                                                          |
                     |                                                          v
                     +-------------------- main.py (menu: input and print) ----+
```

| Module | Job | Uses `input()` / `print()`? | Tested with pytest? |
|--------|-----|------------------------------|---------------------|
| `storage.py` | file lines ⇄ dicts; load and save | no (one warning print for bad lines) | yes |
| `reports.py` | calculations on a list of dicts | **no** | yes |
| `main.py` | the menu and all questions to the user | yes | by running it |

**Why split it this way?** A function that calls `input()` can't be tested automatically: someone has to type.
By keeping all the talking in `main.py`, everything else becomes pure functions that pytest can check in a second.

🤔 **Predict:** which module does each job belong in?
- a) "keep asking until the amount is a number"
- b) "add up the amounts per category"
- c) "turn `2026-11-03,food,Lunch,12.50` into a dict"

Answers: a) `main.py` (it talks to the user) · b) `reports.py` · c) `storage.py`.

## 2. Worked example: one line ⇄ one dict

Our reading log has one line per reading session: `date,book,pages`. The file is `reading_log.txt` in this folder.

The first job of a storage module is to turn **one line** into **one dict**, and back.

In [ ]:
def parse_entry(line):
    """Turn "2026-11-02,Atomic Habits,35" into a dict. Raise ValueError for a bad line."""
    parts = line.strip().split(",")
    if len(parts) != 3:
        raise ValueError(f"expected 3 values, got {len(parts)}")
    return {"date": parts[0].strip(), "book": parts[1].strip(), "pages": int(parts[2])}


def format_entry(entry):
    """Turn an entry dict back into one line of text (without the newline)."""
    return f"{entry['date']},{entry['book']},{entry['pages']}"

In [ ]:
entry = parse_entry("2026-11-02,Atomic Habits,35\n")
print(entry)
print(format_entry(entry))

A great check for this pair: **format then parse gives back the same dict** (a "round trip").

In [ ]:
print(parse_entry(format_entry(entry)) == entry)

🤔 **Predict:** what happens with a line that has only two values? And with `lots` instead of a number?

In [ ]:
parse_entry("2026-11-09,The Hobbit")

In [ ]:
parse_entry("2026-11-09,The Hobbit,lots")

Both raise `ValueError`: one from our own `raise`, one from `int()`. That is useful: the caller can catch
**one** kind of error for every kind of bad line.

## 3. Loading a whole file without crashing

`reading_log.txt` has 6 lines and one of them is broken. One bad line must not crash the whole program, so we
skip it and warn (Day 25).

In [ ]:
from pathlib import Path


def load_entries(path):
    """Return all entries in the file. Missing file -> []. Bad lines are skipped with a warning."""
    path = Path(path)
    if not path.exists():
        return []
    entries = []
    with open(path, encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if line.strip() == "":
                continue
            try:
                entries.append(parse_entry(line))
            except ValueError as error:
                print(f"Warning: skipped line {line_number}: {error}")
    return entries

In [ ]:
log = load_entries("reading_log.txt")
print(len(log), "entries loaded")
print(load_entries("no_such_file.txt"))

## 4. Reports: calculations on a list of dicts

Report functions take the list and **return** a result. No files, no input, no printing.

In [ ]:
def pages_by_book(entries):
    """Return {book: total pages}."""
    totals = {}
    for entry in entries:
        totals[entry["book"]] = totals.get(entry["book"], 0) + entry["pages"]
    return totals


def entries_in_month(entries, month):
    """Return the entries whose date is in month ("YYYY-MM")."""
    return [entry for entry in entries if entry["date"].startswith(month + "-")]


print(pages_by_book(log))
print(len(entries_in_month(log, "2026-11")), "sessions in November")

🤔 **Predict:** why `month + "-"`? What would `startswith("2026-1")` match without it?

In [ ]:
print([entry["date"] for entry in log if entry["date"].startswith("2026-1")])
print(entries_in_month(log, "2026-1"))

Without the `-`, `"2026-1"` matches both October **and** November. Small detail, real bug, and a perfect
**edge-case test**.

## 5. Tests for the worked example

In the project, tests live in `test_storage.py` and `test_reports.py` and run with `uv run pytest`. Here we call
them by hand. Notice: the tests use **small strings and lists**, not the real data file.

In [ ]:
import pytest


def test_parse_entry_normal():
    expected = {"date": "2026-11-02", "book": "Atomic Habits", "pages": 35}
    assert parse_entry("2026-11-02,Atomic Habits,35") == expected


def test_parse_entry_bad_pages_raises():
    with pytest.raises(ValueError):
        parse_entry("2026-11-09,The Hobbit,lots")


def test_pages_by_book():
    entries = [{"date": "2026-11-02", "book": "Dune", "pages": 10},
               {"date": "2026-11-03", "book": "Dune", "pages": 15}]
    assert pages_by_book(entries) == {"Dune": 25}


def test_entries_in_month_does_not_mix_up_months():
    entries = [{"date": "2026-10-28", "book": "Sapiens", "pages": 30}]
    assert entries_in_month(entries, "2026-1") == []


test_parse_entry_normal()
test_parse_entry_bad_pages_raises()
test_pages_by_book()
test_entries_in_month_does_not_mix_up_months()
print("4 tests passed")

## 6. Asking for a number until it is valid

This belongs in `main.py`. The loop only ends with a `return` of a valid number (Day 25's retry loop).

In [ ]:
def ask_pages():
    """Ask until the user types a whole number greater than 0."""
    while True:
        text = input("Pages read: ").strip()
        try:
            pages = int(text)
        except ValueError:
            print("  Please type a whole number, e.g. 25")
        else:
            if pages > 0:
                return pages
            print("  Pages must be more than 0.")


pages = ask_pages()
print("You read", pages, "pages")

## 7. "Press Enter for today"

`date.today()` (Day 26) gives today's date; `.isoformat()` turns it into the same `YYYY-MM-DD` text as the file.

In [ ]:
from datetime import date


def ask_date():
    """Ask for a date as YYYY-MM-DD; Enter means today."""
    while True:
        text = input("Date (YYYY-MM-DD, Enter for today): ").strip()
        if text == "":
            return date.today().isoformat()
        try:
            return date.fromisoformat(text).isoformat()
        except ValueError:
            print("  Not a valid date. Example: 2026-11-16")


print("Date:", ask_date())

## 8. Your plan for today

1. Open `projects/03-expense-tracker/starter/` and read the TODOs in `storage.py`.
2. Run `uv run pytest` there: **4 failed** is the starting point.
3. Build in the order of the README's build plan: storage → reports → safe input → summary → table.
   **Run the tests after every TODO.**
4. Add your own tests as you go (at least 2, one with `pytest.raises`).

Expense lines have **4** values and the amount is a `float`, not an `int`. Do not copy this notebook; adapt it.

**Checkpoint 6** takes the last 15 minutes of the session.